## Steps in Process:
1. First we need to gather all stocks offering a dividend above $2.00 with an ex_dividend date > today
2. Merge response data into a data frame
3. We then want to asyncrhonousely get all ticker data on these dividends to obtain last close and volume
4. If volume on any of the dividend stocks is < 500,000, remove data from results
5. Get all prior dividends on identified stocks with ex_dividend.lt today
6. Get the prior business data along with candles to compare ex and prior performance
7. Add prior business data for day before ex-dividend and ex-dividend to dataframe
8. Subtract day prior to ex and ex-dividend day close prices to observe directional movement in stock

### Steps 1 and 2: gather all stocks offering a dividend above $2.00 with an ex_dividend date > today and merge to dataframe

In [1]:
import requests 
import pandas as pd
from datetime import date

today = date.today() #set todays date
headers = {'Authorization': 'Bearer xxx'} #authentication routing for Polygon API

dividend_url = 'https://api.polygon.io/v3/reference/dividends' #polygon api endpoint for dividends
#set payload and sort by cash amount with greater than $2.00
dividend_payload = {
    'ex_dividend_date.gt': today,
    'cash_amount.gt': 1,
    'limit': 1000,
    'sort': 'cash_amount',
    
} 

r = requests.get(dividend_url, headers=headers, params=dividend_payload).json() #get the response object
df = pd.DataFrame(r['results'])
df.sort_values(['ex_dividend_date'], inplace=True)
df = df[['ticker', 'cash_amount', 'dividend_type','declaration_date','ex_dividend_date','record_date','pay_date']]
df = df.round({'last_close': 2, 'cash_amount': 2})
df

,ticker,cash_amount,dividend_type,declaration_date,ex_dividend_date,record_date,pay_date
22,TXN,1.30,CD,2023-09-22,2023-10-30,2023-10-31,2023-11-14
5,VRTS,1.90,CD,2023-08-16,2023-10-30,2023-10-31,2023-11-15
12,ASML,1.53,CD,2023-10-18,2023-11-01,2023-11-02,2023-11-10
19,NSC,1.35,CD,2023-10-24,2023-11-02,2023-11-03,2023-11-20
39,COST,1.02,CD,2023-10-18,2023-11-02,2023-11-03,2023-11-17
0,ALX,4.50,CD,2023-10-25,2023-11-03,2023-11-06,2023-11-17
36,DKL,1.04,CD,2023-10-25,2023-11-03,2023-11-06,2023-11-13
32,FMX,1.07,CD,2023-09-20,2023-11-03,2023-11-06,2023-11-17
21,AMP,1.35,CD,2023-10-25,2023-11-03,2023-11-06,2023-11-17
30,POOL,1.10,CD,2023-10-25,2023-11-07,2023-11-08,2023-11-22


### Step 3: Asyncrhonousely get all ticker data on these dividends to obtain last close and volume

In [2]:
# https://stackoverflow.com/questions/67944791/fastest-way-to-apply-an-async-function-to-pandas-dataframe
import asyncio
import nest_asyncio
import numpy as np
import pandas as pd

nest_asyncio.apply()

async def ticker_data(x):
    '''
        Async definition to retrieve all ticker data for last close and last volume
    '''
    ticker_url = 'https://api.polygon.io/v2/aggs/ticker/{}/prev?adjusted=true'.format(x)
    ticker_data = requests.get(ticker_url, headers=headers).json()
    last_close = ticker_data['results'][0]['c']
    last_volume = ticker_data['results'][0]['v']
    return last_close, last_volume


async def main():
    x = pd.DataFrame(np.arange(len(df)))
    #zip output of async function to new columns in dataframe
    df['last_close'], df['last_volume'] = zip(*await asyncio.gather(*(ticker_data(x) for x in df['ticker'])))

asyncio.run(main())

### Step 4: If volume on any of the dividend stocks is < 500,000, remove data from results

In [3]:
high_volume = df[df['last_volume'].between(500000, 999999999999)] #filter by volume > 500,000
low_volume = df[df['last_volume'].between(0, 500000)] #identify all low volume stonks

In [4]:
high_volume

,ticker,cash_amount,dividend_type,declaration_date,ex_dividend_date,record_date,pay_date,last_close,last_volume
22,TXN,1.30,CD,2023-09-22,2023-10-30,2023-10-31,2023-11-14,143.12,5327541.0
12,ASML,1.53,CD,2023-10-18,2023-11-01,2023-11-02,2023-11-10,590.00,819484.0
19,NSC,1.35,CD,2023-10-24,2023-11-02,2023-11-03,2023-11-20,184.53,1202106.0
39,COST,1.02,CD,2023-10-18,2023-11-02,2023-11-03,2023-11-17,543.03,1503322.0
32,FMX,1.07,CD,2023-09-20,2023-11-03,2023-11-06,2023-11-17,106.50,859078.0
21,AMP,1.35,CD,2023-10-25,2023-11-03,2023-11-06,2023-11-17,310.56,596864.0
14,URI,1.48,CD,2023-10-25,2023-11-07,2023-11-08,2023-11-22,399.02,933700.0
31,HON,1.08,CD,2023-09-29,2023-11-08,2023-11-10,2023-12-01,177.00,3400371.0
29,TGT,1.10,CD,2023-09-20,2023-11-14,2023-11-15,2023-12-10,107.23,3448150.0
28,LHX,1.14,CD,2023-10-20,2023-11-16,2023-11-17,2023-12-01,170.93,2383643.0


In [5]:
low_volume

,ticker,cash_amount,dividend_type,declaration_date,ex_dividend_date,record_date,pay_date,last_close,last_volume
5,VRTS,1.90,CD,2023-08-16,2023-10-30,2023-10-31,2023-11-15,174.0200,43650.0
0,ALX,4.50,CD,2023-10-25,2023-11-03,2023-11-06,2023-11-17,179.2500,10426.0
36,DKL,1.04,CD,2023-10-25,2023-11-03,2023-11-06,2023-11-13,45.1100,51585.0
30,POOL,1.10,CD,2023-10-25,2023-11-07,2023-11-08,2023-11-22,309.5000,435531.0
15,PH,1.48,CD,2023-10-25,2023-11-09,2023-11-13,2023-12-01,366.2400,476260.0
7,GWW,1.86,CD,2023-10-25,2023-11-10,2023-11-13,2023-12-01,706.7600,409250.0
20,CTAS,1.35,CD,2023-10-24,2023-11-14,2023-11-15,2023-12-15,496.4100,464631.0
1,EQIX,4.26,CD,2023-10-25,2023-11-14,2023-11-15,2023-12-13,710.3900,338549.0
35,PRK,1.05,CD,2023-10-23,2023-11-16,2023-11-17,2023-12-08,100.1200,41635.0
2,AMSF,3.50,SC,2023-10-24,2023-11-30,2023-12-01,2023-12-15,50.2200,65993.0


### Step 5: Get all prior dividends on identified stocks with ex_dividend.lt today

In [6]:
# https://towardsdatascience.com/how-to-convert-json-into-a-pandas-dataframe-100b2ae1e0d8
# https://stackoverflow.com/questions/66864805/json-to-pandas-dataframe-with-nested-lists
# https://stackoverflow.com/questions/952914/how-do-i-make-a-flat-list-out-of-a-list-of-lists

#pd.set_option('display.max_rows', None)

#set the payload to dividends offered less than today
import datetime
'''
    The polygon API starter package only allows 5 years of historical data.
    In order to prevent invalid API responses for dates > 5 years ago, we need to create a new date object
    that represents the current date less five years and limit responses to that from the API request.
    We create that below with the five_years_ago variable.
'''
four_years_ago = date.today() - datetime.timedelta(days=4*365)
four_years_ago = four_years_ago.strftime('%Y-%m-%d')
# x = div_yields[~(div_yields['ex_dividend_date'] < five_years_ago)]

# x
payload = {
    'ticker': '',
    'ex_dividend_date.lte':today, #set maximum date parameter
    'ex_dividend_date.gt': four_years_ago, #set minimum date parameter
    'limit': 1000,
    'sort': 'ex_dividend_date'
}

prior_dividends = [] #empty list for storing dividend data

async def get_prior_dividends(ticker):
    #Loop through each ticker in the dataframe and return results to caller
    url = 'https://api.polygon.io/v3/reference/dividends'
    payload['ticker'] = ticker
    r = requests.get(url, headers=headers, params=payload).json()
    prior_dividends.append(r['results'])


async def main():
    #loop through and asynchronously process each ticker in the dataframe
    await asyncio.gather(*(get_prior_dividends(ticker) for ticker in high_volume['ticker']))

asyncio.run(main())

flat_list = [item for sublist in prior_dividends for item in sublist] #flatten the items for insertion into dataframe
div_yields = pd.DataFrame.from_dict(flat_list)

## Now sort the tickers by dividend date and update dataframe

In [7]:
# pd.set_option('display.max_rows', None)
div_yields = div_yields[['ticker','ex_dividend_date','cash_amount','dividend_type','frequency','declaration_date','record_date','pay_date']]
div_yields = div_yields.round({'cash_amount': 2})
# div_yields

## Class instance to handle stock market holidays and working days

In [8]:
from USActiveStockTrading import USActiveStockTrading
m = USActiveStockTrading()

## Step 6: Get the prior business data along with candles for ex and prior comparison

In [9]:
from datetime import datetime

ts = pd.DataFrame(div_yields[['ticker','cash_amount','ex_dividend_date']]) #create new dataframe of ex_dividend_date
ts['cash_amount'] = ts['cash_amount'].round(decimals=2) #round cash amount to nearest second decimal
#convert values to datetime objects for use in lambda function below
ts['ex_dividend_date'] = pd.to_datetime(ts['ex_dividend_date'])


#get the lst business day before the ex-dividend date
ts['prior_business_date'] = pd.DatetimeIndex(ts.ex_dividend_date) - pd.DateOffset(1)
#convert the ticker value to a string
ts['ticker'] = ts['ticker'].astype(str) 
#convert the prior date value back to a string
ts['prior_business_date'] = ts['prior_business_date'].astype(str)
#convert the ex-dividend date value back to a string
ts['ex_dividend_date'] = ts['ex_dividend_date'].astype(str)
#convert the prior business date value back to a string
ts['prior_business_date'] = ts['prior_business_date'].astype(str)

div_yields = pd.merge(div_yields,ts)
div_yields = div_yields[['ticker','prior_business_date', 'ex_dividend_date','cash_amount','dividend_type','frequency','declaration_date','record_date','pay_date']]

In [10]:
div_yields

,ticker,prior_business_date,ex_dividend_date,cash_amount,dividend_type,frequency,declaration_date,record_date,pay_date
0,TXN,2023-07-27,2023-07-28,1.24,CD,4,2023-07-20,2023-07-31,2023-08-15
1,TXN,2023-05-04,2023-05-05,1.24,CD,4,2023-04-27,2023-05-08,2023-05-16
2,TXN,2023-01-29,2023-01-30,1.24,CD,4,2023-01-19,2023-01-31,2023-02-14
3,TXN,2022-10-27,2022-10-28,1.24,CD,4,2022-09-15,2022-10-31,2022-11-15
4,TXN,2022-07-28,2022-07-29,1.15,CD,4,2022-07-21,2022-08-01,2022-08-16
...,...,...,...,...,...,...,...,...,...
382,ITW,2020-12-29,2020-12-30,1.14,CD,4,2020-10-30,2020-12-31,2021-01-14
383,ITW,2020-09-28,2020-09-29,1.14,CD,4,2020-08-07,2020-09-30,2020-10-14
384,ITW,2020-06-28,2020-06-29,1.07,CD,4,2020-05-08,2020-06-30,2020-07-15
385,ITW,2020-03-29,2020-03-30,1.07,CD,4,2020-02-14,2020-03-31,2020-04-15


### Get all of the dates that are not active trading days

In [11]:
#strip date data of dashes for use with USActiveStockTrading
stp = div_yields.applymap(lambda x: x.replace('-', '') if isinstance(x, str) else x)
#get Boolean list of all dates where market is not open
stp['date_valid'] = [m.is_open(i) for i in stp['prior_business_date']]

#https://stackoverflow.com/questions/54453309/get-index-of-series-where-value-is-true
#https://stackoverflow.com/questions/16729574/how-can-i-get-a-value-from-a-cell-of-a-dataframe

#create index list of all false values
vals = [i for i,j in stp['date_valid'].items() if j==False] 
#get all prior_business dates by index
q = stp.iloc[vals]['prior_business_date']
# q #debugging


In [13]:
#loop through all dates and get last close
corrected_dates = []
for date in q:
    corrected_dates.append(m.last_close(date))
# corrected_dates


In [14]:
#clean up the ugly data in the div_yields dataframe
def replace_bad_dates(x):
    div_yields.at[x[0], 'prior_business_date'] = x[1]
    
result = zip(q.index, corrected_dates)
for i in list(result):
    replace_bad_dates(i)

## Get Ticker, prior close and ex-dividend close, then obtain bar data for last close
The goal here is to get the candlestick data on the day before the ex-dividend date as well on the ex-dividend date to use as compairson information on performance over time. Ideally, this will be completed asynchronously to provide scale and speed. Once complete, comparisons on movement up or down on the underlying security on the ex-dividend date can be captured and stored for future review, informing trading decisions.

In [15]:
import asyncio
import aiohttp
from datetime import datetime

import requests

prior_biz_close = []
xdiv_close = []
to_drop = []

async def get_history_async(data, idx, session):
    url = 'https://api.polygon.io/v2/aggs/ticker/{}/range/1/day/{}/{}?sort=dsc&limit=10'.format(data[0], data[1], data[2])
    async with session.get(url, headers=headers, verify_ssl=False) as resp:
        r = await resp.json()
        # print(idx)
        # print(data)
        # print(r)
        try:
            if (r['queryCount'] < 2) or (r == None) or (r['status'] =='NOT_AUTHORIZED'):
                to_drop.append(idx)
                prior_biz_close.append(0.00) #first data set in results is prior date; get only close
                xdiv_close.append(0.00) #second data set in results is ex-dividend date get only close
                pass
            else:
                prior_biz_close.append(r['results'][0]['c']) #first data set in results is prior date; get only close
                xdiv_close.append(r['results'][1]['c']) #second data set in results is ex-dividend date get only close
        except KeyError:
            pass
            

async def main():
    result = [(ticker, prior_business_date, ex_dividend_date) for (ticker, prior_business_date, ex_dividend_date) in
              zip(div_yields['ticker'], div_yields['prior_business_date'], div_yields['ex_dividend_date'])]

    # asynchronous
    # print(f"{datetime.utcnow()} async start")
    session = aiohttp.ClientSession(trust_env=True)
    tasks = []
    for i in result:
        tasks.append(get_history_async(list(i), result.index(i), session))
    await asyncio.gather(*tasks)
    await session.close()
    # print(f"{datetime.utcnow()} async end")

asyncio.run(main())

In [16]:
div_yields

,ticker,prior_business_date,ex_dividend_date,cash_amount,dividend_type,frequency,declaration_date,record_date,pay_date
0,TXN,2023-07-27,2023-07-28,1.24,CD,4,2023-07-20,2023-07-31,2023-08-15
1,TXN,2023-05-04,2023-05-05,1.24,CD,4,2023-04-27,2023-05-08,2023-05-16
2,TXN,2023-01-27,2023-01-30,1.24,CD,4,2023-01-19,2023-01-31,2023-02-14
3,TXN,2022-10-27,2022-10-28,1.24,CD,4,2022-09-15,2022-10-31,2022-11-15
4,TXN,2022-07-28,2022-07-29,1.15,CD,4,2022-07-21,2022-08-01,2022-08-16
...,...,...,...,...,...,...,...,...,...
382,ITW,2020-12-29,2020-12-30,1.14,CD,4,2020-10-30,2020-12-31,2021-01-14
383,ITW,2020-09-28,2020-09-29,1.14,CD,4,2020-08-07,2020-09-30,2020-10-14
384,ITW,2020-06-26,2020-06-29,1.07,CD,4,2020-05-08,2020-06-30,2020-07-15
385,ITW,2020-03-27,2020-03-30,1.07,CD,4,2020-02-14,2020-03-31,2020-04-15


In [17]:
# div_yields['prior_biz_close']
div_yields

,ticker,prior_business_date,ex_dividend_date,cash_amount,dividend_type,frequency,declaration_date,record_date,pay_date
0,TXN,2023-07-27,2023-07-28,1.24,CD,4,2023-07-20,2023-07-31,2023-08-15
1,TXN,2023-05-04,2023-05-05,1.24,CD,4,2023-04-27,2023-05-08,2023-05-16
2,TXN,2023-01-27,2023-01-30,1.24,CD,4,2023-01-19,2023-01-31,2023-02-14
3,TXN,2022-10-27,2022-10-28,1.24,CD,4,2022-09-15,2022-10-31,2022-11-15
4,TXN,2022-07-28,2022-07-29,1.15,CD,4,2022-07-21,2022-08-01,2022-08-16
...,...,...,...,...,...,...,...,...,...
382,ITW,2020-12-29,2020-12-30,1.14,CD,4,2020-10-30,2020-12-31,2021-01-14
383,ITW,2020-09-28,2020-09-29,1.14,CD,4,2020-08-07,2020-09-30,2020-10-14
384,ITW,2020-06-26,2020-06-29,1.07,CD,4,2020-05-08,2020-06-30,2020-07-15
385,ITW,2020-03-27,2020-03-30,1.07,CD,4,2020-02-14,2020-03-31,2020-04-15


In [48]:
div_yields[div_yields['ticker'] == 'TGT']

,ticker,prior_business_date,ex_dividend_date,cash_amount,dividend_type,frequency,declaration_date,record_date,pay_date,prior_biz_close,xdiv_close,delta
0,TGT,2023-07-27,2023-07-28,1.24,CD,4,2023-07-20,2023-07-31,2023-08-15,162.30,165.82,3.52
1,TGT,2023-05-04,2023-05-05,1.24,CD,4,2023-04-27,2023-05-08,2023-05-16,177.72,178.37,0.65
2,TGT,2023-01-27,2023-01-30,1.24,CD,4,2023-01-19,2023-01-31,2023-02-14,175.24,173.13,-2.11
3,TGT,2022-10-27,2022-10-28,1.24,CD,4,2022-09-15,2022-10-31,2022-11-15,156.76,161.36,4.60
4,TGT,2022-07-28,2022-07-29,1.15,CD,4,2022-07-21,2022-08-01,2022-08-16,175.75,178.89,3.14
...,...,...,...,...,...,...,...,...,...,...,...,...
382,TGT,2020-12-29,2020-12-30,1.14,CD,4,2020-10-30,2020-12-31,2021-01-14,387.04,388.91,1.87
383,TGT,2020-09-28,2020-09-29,1.14,CD,4,2020-08-07,2020-09-30,2020-10-14,249.45,244.77,-4.68
384,TGT,2020-06-26,2020-06-29,1.07,CD,4,2020-05-08,2020-06-30,2020-07-15,265.02,263.85,-1.17
385,TGT,2020-03-27,2020-03-30,1.07,CD,4,2020-02-14,2020-03-31,2020-04-15,189.83,189.25,-0.58


## Step 7: Add the prior business close and ex-dividend close data to the dataframe

In [19]:
div_yields['prior_biz_close'] = prior_biz_close
div_yields['xdiv_close'] = xdiv_close
# len(div_yields) #50
# len(xdiv_close) #41
# len(prior_biz_close) #41

In [20]:
#remove all instances of rows where column value is 0.00
div_yields = div_yields[div_yields.prior_biz_close != 0]
# div_yields

## Step 8: Subtract the ex-dividend date from the prior business close to find the delta

In [21]:
div_yields['delta'] =  div_yields['xdiv_close'] - div_yields['prior_biz_close']
# div_yields

/var/folders/wk/zgdj52nx75gcms8jvb82chd00000gp/T/ipykernel_88321/887779326.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  div_yields['delta'] =  div_yields['xdiv_close'] - div_yields['prior_biz_close']


In [22]:
#https://www.kite.com/blog/python/pandas-groupby-count-value-count/
import numpy as np
v = div_yields.groupby('ticker')

v['delta'].agg(np.mean)
b = pd.DataFrame(v['delta'].agg(np.mean))
b

,delta
ticker,
AMGN,-1.261875
AMP,-0.371250
ASML,-1.677273
CCI,-3.556000
CI,-3.130000
CMI,0.341875
COST,-0.801176
DPZ,-1.655000
DUK,-1.659375


In [23]:
g = div_yields.groupby('ticker')['delta']
l = pd.DataFrame(g.agg(
    pos_count=lambda s: s.gt(0).sum(),
    neg_count=lambda s: s.lt(0).sum(),
    net_count=lambda s: s.gt(0).sum() + s.lt(0).sum()).astype(int))
    # open_down=lambda s: pos_count/net_count
t = pd.concat([b,l], axis=1)
t['decrease_liklihood_%'] = (t['neg_count']/t['net_count'])*100 
t['increase_liklihood_%'] = (t['pos_count']/t['net_count'])*100
t.reset_index()

,ticker,delta,pos_count,neg_count,net_count,decrease_liklihood_%,increase_liklihood_%
0,AMGN,-1.261875,5,11,16,68.750000,31.250000
1,AMP,-0.371250,5,11,16,68.750000,31.250000
2,ASML,-1.677273,5,6,11,54.545455,45.454545
3,CCI,-3.556000,3,12,15,80.000000,20.000000
4,CI,-3.130000,3,8,11,72.727273,27.272727
5,CMI,0.341875,5,11,16,68.750000,31.250000
6,COST,-0.801176,6,11,17,64.705882,35.294118
7,DPZ,-1.655000,5,11,16,68.750000,31.250000
8,DUK,-1.659375,2,14,16,87.500000,12.500000
9,ELV,-1.750000,9,6,15,40.000000,60.000000


In [24]:
# https://www.geeksforgeeks.org/how-to-sum-negative-and-positive-values-using-groupby-in-pandas/
def pos(col): 
  return col[col > 0].mean()
  
def neg(col): 
  return col[col < 0].mean()

# print(['Y'].agg([('negative_values', neg),
#                   ('positive_values', pos)
#                   ]))

w = div_yields.groupby(div_yields['ticker'])

s = pd.DataFrame(w['delta'].agg([('average_increase', neg),
                  ('average_decrease', pos)
                  ]))
s = s.fillna(0)
s = abs(s)
s['average_decrease'] = -s['average_decrease']
s = pd.concat([t,s], axis=1)
s = s.round({'delta': 2, 'average_increase': 2, 'average_decrease': 2})
s = s[['pos_count', 'neg_count', 'net_count', 'decrease_liklihood_%', 'average_decrease', 'increase_liklihood_%','average_increase','delta']]
s

,pos_count,neg_count,net_count,decrease_liklihood_%,average_decrease,increase_liklihood_%,average_increase,delta
ticker,,,,,,,,
AMGN,5,11,16,68.750000,-1.46,31.250000,2.50,-1.26
AMP,5,11,16,68.750000,-3.13,31.250000,1.96,-0.37
ASML,5,6,11,54.545455,-3.38,45.454545,5.89,-1.68
CCI,3,12,15,80.000000,-2.86,20.000000,5.16,-3.56
CI,3,8,11,72.727273,-2.38,27.272727,5.20,-3.13
CMI,5,11,16,68.750000,-5.28,31.250000,1.90,0.34
COST,6,11,17,64.705882,-1.29,35.294118,1.94,-0.80
DPZ,5,11,16,68.750000,-1.43,31.250000,3.06,-1.66
DUK,2,14,16,87.500000,-1.26,12.500000,2.08,-1.66


In [25]:
high_volume

,ticker,cash_amount,dividend_type,declaration_date,ex_dividend_date,record_date,pay_date,last_close,last_volume
22,TXN,1.30,CD,2023-09-22,2023-10-30,2023-10-31,2023-11-14,143.12,5327541.0
12,ASML,1.53,CD,2023-10-18,2023-11-01,2023-11-02,2023-11-10,590.00,819484.0
19,NSC,1.35,CD,2023-10-24,2023-11-02,2023-11-03,2023-11-20,184.53,1202106.0
39,COST,1.02,CD,2023-10-18,2023-11-02,2023-11-03,2023-11-17,543.03,1503322.0
32,FMX,1.07,CD,2023-09-20,2023-11-03,2023-11-06,2023-11-17,106.50,859078.0
21,AMP,1.35,CD,2023-10-25,2023-11-03,2023-11-06,2023-11-17,310.56,596864.0
14,URI,1.48,CD,2023-10-25,2023-11-07,2023-11-08,2023-11-22,399.02,933700.0
31,HON,1.08,CD,2023-09-29,2023-11-08,2023-11-10,2023-12-01,177.00,3400371.0
29,TGT,1.10,CD,2023-09-20,2023-11-14,2023-11-15,2023-12-10,107.23,3448150.0
28,LHX,1.14,CD,2023-10-20,2023-11-16,2023-11-17,2023-12-01,170.93,2383643.0


In [42]:
high_volume = high_volume[['ticker','last_close','last_volume', 'cash_amount', 'dividend_type','ex_dividend_date','record_date','pay_date']]
o = pd.merge(high_volume, s, how="outer", on=["ticker"])
# o = o.fillna('-')
o['div%'] = (o['cash_amount']/o['last_close'])*100
o = o.round({'last_close': 2, 'cash_amount': 2, 'div%': 2})
o = o.rename(columns={
    'dividend_type': 'type', 
    'pos_count': '#+', 
    'neg_count': '#-', 
    'net_count': 'total', 
    'decrease_liklihood_%': '↓µ%', 
    'average_decrease': '↓µ$',
    'increase_liklihood_%': '↑µ%',
    'average_increase': '↑µ$',
    'delta': '∆'
})
o.sort_values(['ex_dividend_date'], inplace=True)
o = o.round({'↓µ%': 0, '↑µ%': 0})
o = o[['ticker', 'last_close', 'last_volume', 'cash_amount', 'div%', 'type', 'ex_dividend_date', 'record_date', 'pay_date', '#+', '#-', 'total', '↓µ%', '↓µ$', '↑µ%', '↑µ$', '∆']]
o
# o[o['volume'].between(0, 500000)] #filter by volume > 500,000


,ticker,last_close,last_volume,cash_amount,div%,type,ex_dividend_date,record_date,pay_date,#+,#-,total,↓µ%,↓µ$,↑µ%,↑µ$,∆
0,TXN,143.12,5327541.0,1.30,0.91,CD,2023-10-30,2023-10-31,2023-11-14,8.0,8.0,16.0,50.0,-3.14,50.0,3.49,-0.18
1,ASML,590.00,819484.0,1.53,0.26,CD,2023-11-01,2023-11-02,2023-11-10,5.0,6.0,11.0,55.0,-3.38,45.0,5.89,-1.68
2,NSC,184.53,1202106.0,1.35,0.73,CD,2023-11-02,2023-11-03,2023-11-20,8.0,8.0,16.0,50.0,-3.57,50.0,3.27,0.15
3,COST,543.03,1503322.0,1.02,0.19,CD,2023-11-02,2023-11-03,2023-11-17,6.0,11.0,17.0,65.0,-1.29,35.0,1.94,-0.80
4,FMX,106.50,859078.0,1.07,1.00,CD,2023-11-03,2023-11-06,2023-11-17,4.0,4.0,8.0,50.0,-4.30,50.0,4.02,0.14
5,AMP,310.56,596864.0,1.35,0.43,CD,2023-11-03,2023-11-06,2023-11-17,5.0,11.0,16.0,69.0,-3.13,31.0,1.96,-0.37
6,URI,399.02,933700.0,1.48,0.37,CD,2023-11-07,2023-11-08,2023-11-22,1.0,2.0,3.0,67.0,-3.38,33.0,3.21,-1.01
7,HON,177.00,3400371.0,1.08,0.61,CD,2023-11-08,2023-11-10,2023-12-01,9.0,7.0,16.0,44.0,-3.33,56.0,4.78,-0.22
8,TGT,107.23,3448150.0,1.10,1.03,CD,2023-11-14,2023-11-15,2023-12-10,2.0,12.0,14.0,86.0,-7.43,14.0,1.60,-0.29
15,HSY,184.11,2067056.0,1.19,0.65,CD,2023-11-16,2023-11-17,2023-12-15,2.0,13.0,15.0,87.0,-1.12,13.0,2.55,-2.06


In [44]:
#get only values where the mean percentage decline in stock value the day after ex-dividend is greater than 70%
targets = o[o['↓µ%'] > 70.0]
targets


,ticker,last_close,last_volume,cash_amount,div%,type,ex_dividend_date,record_date,pay_date,#+,#-,total,↓µ%,↓µ$,↑µ%,↑µ$,∆
8,TGT,107.23,3448150.0,1.10,1.03,CD,2023-11-14,2023-11-15,2023-12-10,2.0,12.0,14.0,86.0,-7.43,14.0,1.60,-0.29
15,HSY,184.11,2067056.0,1.19,0.65,CD,2023-11-16,2023-11-17,2023-12-15,2.0,13.0,15.0,87.0,-1.12,13.0,2.55,-2.06
14,WHR,102.10,2931467.0,1.75,1.71,CD,2023-11-16,2023-11-17,2023-12-15,4.0,12.0,16.0,75.0,-1.64,25.0,1.82,-0.95
13,SJM,112.75,1274023.0,1.06,0.94,CD,2023-11-16,2023-11-17,2023-12-01,4.0,12.0,16.0,75.0,-1.51,25.0,1.99,-1.11
10,DUK,87.53,2764436.0,1.02,1.17,CD,2023-11-16,2023-11-17,2023-12-18,2.0,14.0,16.0,88.0,-1.26,12.0,2.08,-1.66
9,LHX,170.93,2383643.0,1.14,0.67,CD,2023-11-16,2023-11-17,2023-12-01,3.0,12.0,15.0,80.0,-1.84,20.0,3.68,-2.57
11,PSX,110.93,5214890.0,1.05,0.95,CD,2023-11-16,2023-11-17,2023-12-01,4.0,12.0,16.0,75.0,-1.76,25.0,1.76,-0.88
16,JNJ,145.60,11385751.0,1.19,0.82,CD,2023-11-20,2023-11-21,2023-12-05,1.0,13.0,14.0,93.0,-1.09,7.0,6.14,-5.62
19,MCD,255.76,3209027.0,1.67,0.65,CD,2023-11-30,2023-12-01,2023-12-15,4.0,12.0,16.0,75.0,-5.82,25.0,3.24,-0.97
21,LIN,370.53,1945339.0,1.27,0.34,CD,2023-12-01,2023-12-04,2023-12-18,2.0,5.0,7.0,71.0,-3.08,29.0,3.69,-1.76


In [ ]:
'''
To Do:
1. get historical options price history for three days before and following ex-dividend date
2. Analyze average movement up/down in options price in relation to rise/decline of stock
3. Get historical underlying stock price movement three days before and after for comparison
4. Add average option price adjusted for dividend event

See: https://polygon.io/docs/options/get_v2_aggs_ticker__optionsticker__range__multiplier___timespan___from___to for details
'''